# 00 — Funções de features da corrida

Notebook auxiliar carregado com `%run ./00_features_corrida` pelos notebooks 09 e 11. Centraliza o cálculo das features para que **treino e inferência usem exatamente a mesma regra**, inclusive dentro do modelo publicado no Model Serving.

| Feature | Regra | Disponível antes da corrida? |
|---|---|---|
| `distancia_lalo` | Haversine entre origem e destino (raio 6371,0088 km, igual à biblioteca `haversine` do `02_silver`) | Sim |
| `hora` | Hora da **solicitação** | Sim |
| `dia_semana` | Dia da semana no padrão Spark `dayofweek` (1 = domingo … 7 = sábado) | Sim |
| `periodo` | 1 = 07h–11h; 2 = 12h–17h; 3 = demais (mesma regra do `02_silver`) | Sim |

No histórico só existe `datahora_entregue`. O horário da solicitação é aproximado por **`datahora_inicio = datahora_entregue − duracao`**. Assim as features de calendário deixam de usar informação que só existe depois da corrida, o vazamento temporal apontado no notebook 08.

In [ ]:
import math
import numpy as np
import pandas as pd

RAIO_TERRA_KM = 6371.0088
FEATURES_DURACAO = ["distancia_lalo", "hora", "dia_semana", "periodo"]
FEATURES_PRECO = ["distancia_lalo", "hora", "dia_semana", "periodo", "duracao_estimada"]
COLUNAS_ENTRADA = ["latitude_origem", "longitude_origem", "latitude_destino", "longitude_destino", "datahora_solicitacao"]


def haversine_km(lat_o, lon_o, lat_d, lon_d):
    """Distância em linha reta (km), vetorizada."""
    lat_o, lon_o, lat_d, lon_d = map(np.radians, (lat_o, lon_o, lat_d, lon_d))
    a = np.sin((lat_d - lat_o) / 2) ** 2 + np.cos(lat_o) * np.cos(lat_d) * np.sin((lon_d - lon_o) / 2) ** 2
    return 2 * RAIO_TERRA_KM * np.arcsin(np.sqrt(a))


def calcular_periodo(hora):
    hora = np.asarray(hora)
    return np.select([(hora >= 7) & (hora <= 11), (hora >= 12) & (hora <= 17)], [1, 2], 3)


def preparar_features(df, coluna_datahora="datahora_solicitacao"):
    """Recebe coordenadas + data/hora da solicitação e devolve as features do modelo."""
    datahora = pd.to_datetime(df[coluna_datahora])
    saida = pd.DataFrame(index=df.index)
    saida["distancia_lalo"] = haversine_km(df["latitude_origem"].astype(float), df["longitude_origem"].astype(float),
                                           df["latitude_destino"].astype(float), df["longitude_destino"].astype(float))
    saida["hora"] = datahora.dt.hour.astype(int)
    saida["dia_semana"] = ((datahora.dt.dayofweek + 1) % 7 + 1).astype(int)
    saida["periodo"] = calcular_periodo(saida["hora"]).astype(int)
    return saida

In [ ]:
# Teste rápido: segunda-feira 08:15 → dia_semana 2, período 1
_exemplo = pd.DataFrame([{"latitude_origem": -23.5614, "longitude_origem": -46.6559,
                          "latitude_destino": -23.5874, "longitude_destino": -46.6576,
                          "datahora_solicitacao": "2026-11-02 08:15:00"}])
display(preparar_features(_exemplo))